# From Policy Gradient to RLHF (PPO)

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/reinforcement-learning/05-from-policy-gradient-to-rlhf

We'll run vanilla REINFORCE and clipped PPO on the same toy 2-arm bandit. With a high-variance advantage, vanilla policy gradient can blow the policy off the cliff in a single step. PPO's clipped surrogate caps how far each update can drift — same gradient direction, bounded step.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## A 2-arm bandit

Two actions. Arm 1 has a higher *mean* reward, but both arms are very noisy so the realised advantage of arm 1 over arm 0 is a high-variance signal. That's exactly the regime where vanilla policy gradient struggles.

In [ ]:
MEANS = np.array([0.0, 1.0])   # arm 1 is better
STD = 4.0                       # very noisy → high-variance advantages

def softmax(z):
    z = z - z.max()
    e = np.exp(z); return e/e.sum()

def sample(rng, theta):
    p = softmax(theta)
    a = rng.choice(2, p=p)
    r = MEANS[a] + STD * rng.standard_normal()
    return a, r, p

rng = np.random.default_rng(0)
theta = np.zeros(2)
print('initial policy:', softmax(theta))

## Vanilla REINFORCE on noisy advantages

$\theta \leftarrow \theta + \alpha \cdot A \cdot \nabla_\theta \log \pi(a)$. With $A$ this noisy, a single batch with a huge negative realised advantage on arm 1 can shove the policy hard toward arm 0 — even though arm 1 is the better arm in expectation.

In [ ]:
def reinforce_run(seed, steps=400, lr=0.2, batch=8):
    rng = np.random.default_rng(seed)
    theta = np.zeros(2); history = []
    for _ in range(steps):
        grad = np.zeros(2)
        # collect a small batch, compute advantages with the batch baseline
        actions, rewards, probs0 = [], [], None
        for _ in range(batch):
            a, r, p = sample(rng, theta)
            if probs0 is None: probs0 = p
            actions.append(a); rewards.append(r)
        baseline = np.mean(rewards)
        for a, r in zip(actions, rewards):
            adv = r - baseline
            one_hot = np.zeros(2); one_hot[a] = 1.0
            grad += adv * (one_hot - probs0)
        theta = theta + lr * grad / batch
        history.append(softmax(theta)[1])
    return np.array(history)

vanilla = np.stack([reinforce_run(seed=s) for s in range(20)])
print('vanilla REINFORCE: final P(arm 1) mean =', vanilla[:, -1].mean().round(3),
      ' std =', vanilla[:, -1].std().round(3))

## PPO on the same bandit

At the start of each update, snapshot $\pi_{\theta_\text{old}}$. Compute the importance ratio $\rho = \pi_\theta(a) / \pi_{\theta_\text{old}}(a)$. Take **gradient steps on the clipped surrogate** $\min(\rho A,\ \text{clip}(\rho, 1-\epsilon, 1+\epsilon)\cdot A)$. The clip caps how far one batch can push the policy regardless of how noisy the advantages are.

In [ ]:
def ppo_run(seed, steps=400, lr=0.2, batch=8, epsilon=0.2, inner_epochs=4):
    rng = np.random.default_rng(seed)
    theta = np.zeros(2); history = []
    for _ in range(steps):
        # rollout under the snapshot policy
        theta_old = theta.copy()
        actions, rewards = [], []
        for _ in range(batch):
            a, r, _ = sample(rng, theta_old)
            actions.append(a); rewards.append(r)
        baseline = np.mean(rewards)
        advs = np.array([r - baseline for r in rewards])
        # multiple gradient steps on the same batch, clipped
        for _ in range(inner_epochs):
            grad = np.zeros(2)
            p = softmax(theta); p_old = softmax(theta_old)
            for a, A in zip(actions, advs):
                rho = p[a] / p_old[a]
                # the clip kills the gradient when (A>0 and ρ>1+ε) or (A<0 and ρ<1-ε)
                if (A > 0 and rho > 1 + epsilon) or (A < 0 and rho < 1 - epsilon):
                    continue
                one_hot = np.zeros(2); one_hot[a] = 1.0
                grad += rho * A * (one_hot - p)
            theta = theta + lr * grad / batch
        history.append(softmax(theta)[1])
    return np.array(history)

ppo = np.stack([ppo_run(seed=s) for s in range(20)])
print('PPO: final P(arm 1) mean =', ppo[:, -1].mean().round(3),
      ' std =', ppo[:, -1].std().round(3))

## Vanilla blows up; PPO stays bounded

Plot $P(\text{arm 1})$ over time across 20 seeds. Vanilla policy gradient sees a few large negative advantages early and some runs collapse onto the wrong arm. PPO's clip absorbs the same noise — every seed converges to the better arm.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for run in vanilla:
    ax[0].plot(run, color='#f59e0b', alpha=0.4, linewidth=1)
ax[0].plot(vanilla.mean(0), color='#fbbf24', linewidth=2.5, label='mean')
ax[0].set_title('Vanilla REINFORCE'); ax[0].set_xlabel('update'); ax[0].set_ylabel('P(arm 1)')
ax[0].axhline(1.0, color='#22c55e', linestyle='--', alpha=0.4); ax[0].legend()
for run in ppo:
    ax[1].plot(run, color='#6366f1', alpha=0.4, linewidth=1)
ax[1].plot(ppo.mean(0), color='#a5b4fc', linewidth=2.5, label='mean')
ax[1].set_title('PPO (clipped, ε=0.2)'); ax[1].set_xlabel('update')
ax[1].axhline(1.0, color='#22c55e', linestyle='--', alpha=0.4); ax[1].legend()
plt.suptitle('Same bandit, same advantages — clip changes everything', color='#e2e8f0')
plt.show()

## RLHF connection in one line

Stretch one variable: replace `sample()` with sampling a token from $\pi_\theta(y_t | x, y_{<t})$, and replace `reward` with `r_φ(x, y) · 1[t=T]  −  β · log(π_θ(y_t)/π_ref(y_t))`. That's RLHF. The PPO loop above is unchanged.

## Key takeaways

- Vanilla policy gradient is unbiased but high-variance; a single bad batch can ruin the policy.
- PPO clips the surrogate to $[1-\epsilon, 1+\epsilon]$ — the gradient vanishes past the bound, so each update has a soft trust region.
- The same loop, with a reward model at end-of-sequence and a per-token KL leash, is exactly **RLHF**.

## ✏️ Your turn

### Exercise — Implement the clipped surrogate

Given the importance ratio $\rho$ and the advantage $A$, the per-token clipped surrogate is

$$L = \min\!\big(\rho \cdot A,\ \text{clip}(\rho,\, 1-\epsilon,\, 1+\epsilon) \cdot A\big).$$

Write the scalar form. Pay attention to the sign of $A$: the clip is asymmetric in effect depending on whether the advantage is positive or negative.

In [ ]:
def ppo_clip_surrogate(rho, advantage, epsilon=0.2):
    """Per-token PPO clipped surrogate.
    rho:        float, importance ratio π_θ(a)/π_θ_old(a) (>0).
    advantage:  float, A_t.
    epsilon:    float, clip half-width (default 0.2).
    Returns:    float, min(rho*A, clip(rho, 1-eps, 1+eps)*A).
    """
    # TODO(you): compute clipped_rho via min(max(rho, 1-eps), 1+eps); return min of the two terms
    ...

In [ ]:
# At ρ = 1 the two terms are equal → A.
assert abs(ppo_clip_surrogate(1.0, 3.0) - 3.0) < 1e-9, \
    "at ρ=1 the surrogate equals A"
# A > 0 and ρ above the upper band → clipped wins: 1.2 * 2 = 2.4 (vs 1.5 * 2 = 3.0).
assert abs(ppo_clip_surrogate(1.5, 2.0, 0.2) - 2.4) < 1e-9, \
    "A>0, ρ>1+ε: min picks the clipped (smaller) term — caps the upside"
# A < 0 and ρ below the lower band → clipped wins: 0.8 * (-2) = -1.6 (vs 0.5 * (-2) = -1.0).
assert abs(ppo_clip_surrogate(0.5, -2.0, 0.2) - (-1.6)) < 1e-9, \
    "A<0, ρ<1-ε: min picks the clipped (more negative) term — caps the downside"
# A > 0 and ρ below the lower band → unclipped wins (clip doesn't bite when ρ is small).
assert abs(ppo_clip_surrogate(0.5, 2.0, 0.2) - 1.0) < 1e-9, \
    "A>0, ρ<1-ε: unclipped term (0.5*2=1.0) is smaller than clipped (0.8*2=1.6)"
# Zero advantage (analogous to "zero reward everywhere"): both the clipped and
# unclipped terms vanish, regardless of rho -- no learning signal either way.
assert abs(ppo_clip_surrogate(1.5, 0.0, 0.2) - 0.0) < 1e-9, \
    "advantage=0: the surrogate is 0 no matter what rho is"
print("✅ Exercise passed")

<details>
<summary>💡 Show solution</summary>

```python
def ppo_clip_surrogate(rho, advantage, epsilon=0.2):
    clipped_rho = min(max(rho, 1.0 - epsilon), 1.0 + epsilon)
    return min(rho * advantage, clipped_rho * advantage)
```

Why `min` and not `max`? The clipped surrogate is a **pessimistic lower bound** on the policy improvement. PPO maximises this lower bound, which is what makes the clip a one-sided trust region: it stops *rewarding* moves outside the band but doesn't penalise being there.

</details>

### Exercise 2 — Extra practice: the GRPO objective, applied (DML #101)

The full derivation of **GRPO** — group-relative advantages, the clipped surrogate above, and the KL-to-reference penalty — lives in the companion wiki notebook: **[`notebooks/wiki/grpo-objective.ipynb`](https://ml-viz-ruby.vercel.app/wiki/grpo-objective)**. Here we just wire up the one short combination DML asks for, reusing `ppo_clip_surrogate` from Exercise 1 above.

[DML #101 — Implement the GRPO Objective Function](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/101_implement-the-grpo-objective-function) matches this signature:

```python
grpo_objective(rhos, A, pi_theta_old, pi_theta_ref, epsilon=0.2, beta=0.01) -> float
```

It averages the clipped surrogate over a group of $G$ sampled completions, then subtracts $\beta \cdot D_{KL}(\pi_\theta \| \pi_{ref})$, where $\pi_\theta$ is recovered from `rhos * pi_theta_old` (renormalized to sum to 1, same as `pi_theta_ref`).

In [ ]:
def grpo_objective(rhos, A, pi_theta_old, pi_theta_ref, epsilon=0.2, beta=0.01):
    """GRPO objective: mean clipped surrogate minus a KL penalty to the reference policy.
    rhos, A, pi_theta_old, pi_theta_ref: equal-length sequences over a group of G samples.
    Returns: float."""
    # TODO(you): 1) mean of ppo_clip_surrogate(rho, a, epsilon) over the group
    #            2) recover pi_theta = rhos * pi_theta_old, renormalize both
    #               pi_theta and pi_theta_ref to sum to 1
    #            3) KL(pi_theta || pi_theta_ref) = sum(pi_theta * log(pi_theta/pi_theta_ref))
    #            4) return mean_surrogate - beta * kl
    ...

In [ ]:
assert abs(grpo_objective([1.2, 0.8, 1.1], [1.0, 1.0, 1.0], [0.9, 1.1, 1.0], [1.0, 0.5, 1.5],
                          epsilon=0.2, beta=0.01) - 1.032749) < 1e-4, \
    "matches the DML #101 fixture exactly"
assert abs(grpo_objective([0.9, 1.1], [1.0, 1.0], [1.0, 1.0], [0.8, 1.2],
                          epsilon=0.1, beta=0.05) - 0.999743) < 1e-4, \
    "matches a second DML #101 fixture"
assert abs(grpo_objective([1.5, 0.5, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.2, 0.7, 1.3],
                          epsilon=0.15, beta=0.02) - 0.882682) < 1e-4, \
    "matches a third DML #101 fixture"

# Edge case: a group of size 1 (G=1). pi_theta and pi_theta_ref both renormalize
# to [1.0], so the KL term is exactly 0 -- the objective reduces to the surrogate alone.
assert abs(grpo_objective([1.0], [1.0], [1.0], [1.0], epsilon=0.1, beta=0.01) - 1.0) < 1e-9, \
    "G=1: a single-sample group has zero KL to itself, so the objective is just the surrogate"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def grpo_objective(rhos, A, pi_theta_old, pi_theta_ref, epsilon=0.2, beta=0.01):
    rhos = np.asarray(rhos, dtype=float)
    A = np.asarray(A, dtype=float)
    surrogate = np.mean([ppo_clip_surrogate(rho, a, epsilon) for rho, a in zip(rhos, A)])

    pi_theta = rhos * np.asarray(pi_theta_old, dtype=float)
    pi_theta = pi_theta / pi_theta.sum()
    pi_ref = np.asarray(pi_theta_ref, dtype=float)
    pi_ref = pi_ref / pi_ref.sum()
    kl = np.sum(pi_theta * np.log(pi_theta / pi_ref + 1e-10))

    return surrogate - beta * kl
```

</details>